# **Exploratory Data Analysis with SQL**

## Introduction
In this notebook, I'll be:

1. Working with the SpaceX dataset to better understand patterns in launches
2. Loading the dataset into a SQLite database for analysis
3. Executing SQL queries to extract meaningful insights about SpaceX launches


## Overview of the Dataset

SpaceX has gained worldwide attention for a series of historic milestones in private space exploration. It is the only private company ever to return a spacecraft from low-earth orbit, which it first accomplished in December 2010.

SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars whereas other providers cost upward of 165 million dollars each. Much of the savings comes from SpaceX's ability to reuse the first stage of the rocket.

Therefore, if I can determine if the first stage will land successfully, I can estimate the cost of a launch. This information could be valuable if an alternate company wants to bid against SpaceX for a rocket launch contract.

This dataset includes a record for each payload carried during a SpaceX mission into outer space.


### Connect to the database

First, I'll load the SQL extension and establish a connection with the database


In [ ]:
# Install SQL extension if working locally
#!pip install ipython-sql

In [ ]:
%load_ext sql

In [ ]:
import csv, sqlite3

# Create a connection to the SQLite database
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [ ]:
# Install specific pandas version for compatibility
!pip install -q pandas==1.1.5

In [ ]:
# Connect to the SQLite database using SQL magic
%sql sqlite:///my_data1.db

In [ ]:
# Load the SpaceX dataset and save it to the SQLite database
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

**Note: The following code removes blank rows from the table**


In [ ]:
# Create a clean table without null dates
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

## SQL Analysis

Now I'll write and execute SQL queries to analyze the SpaceX launch data.

**Note: For column names in mixed case, I need to enclose them in double quotes**


### Analysis 1: Launch Site Distribution

First, I want to identify all the unique launch sites used in SpaceX missions.


In [ ]:
%%sql
-- Query to find unique launch sites
SELECT DISTINCT Launch_Site
FROM SPACEXTBL;

### Analysis 2: Cape Canaveral Air Force Station Launches

Let me examine launches from sites that begin with "CCA" (Cape Canaveral Air Force Station).


In [ ]:
%%sql
-- Find launches from Cape Canaveral Air Force Station
SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE "CCA%"
LIMIT 5;

### Analysis 3: NASA Payload Mass

I'm interested in the total payload mass carried by boosters for NASA's Commercial Resupply Services missions.


In [ ]:
%%sql
-- Calculate total payload mass for NASA (CRS) missions
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_NASA_Payload_Mass
FROM SPACEXTBL
WHERE Customer = "NASA (CRS)";

### Analysis 4: Falcon 9 v1.1 Payload Capacity

Now I'll examine the average payload mass carried by the Falcon 9 v1.1 booster version.


In [ ]:
%%sql
-- Count launches for each variant of the F9 v1.1 booster
SELECT Booster_Version, COUNT(*) AS Launch_Count
FROM SPACEXTBL
GROUP BY Booster_Version
HAVING Booster_Version LIKE "F9 v1.1%";

In [ ]:
%%sql
-- Calculate average payload mass for the base F9 v1.1 booster
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass
FROM SPACEXTBL
WHERE Booster_Version = "F9 v1.1";

In [ ]:
%%sql
-- Calculate average payload mass for all F9 v1.1 variants
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass_All_Variants
FROM SPACEXTBL
WHERE Booster_Version LIKE "F9 v1.1%";

### Analysis 5: First Successful Ground Pad Landing

I want to identify when SpaceX achieved its first successful landing on a ground pad.


In [ ]:
%%sql
-- Find the date of the first successful ground pad landing
SELECT Date
FROM SPACEXTBL
WHERE Landing_Outcome = "Success (ground pad)"
ORDER BY Date
LIMIT 1;

### Analysis 6: Successful Drone Ship Landings with Medium Payloads

Let me find boosters that successfully landed on drone ships while carrying payloads between 4000kg and 6000kg.


In [ ]:
%%sql
-- Find boosters with successful drone ship landings and medium-weight payloads
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE Landing_Outcome = "Success (drone ship)" AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;

### Analysis 7: Mission Success and Failure Rates

I'll analyze the total number of successful and failed missions.


In [ ]:
%%sql
-- View all possible mission outcomes
SELECT DISTINCT Mission_Outcome
FROM SPACEXTBL;

In [ ]:
%%sql
-- Count successful missions
SELECT COUNT(*) AS Total_Success
FROM SPACEXTBL
WHERE Mission_Outcome LIKE "%Success%";

In [ ]:
%%sql
-- Count failed missions
SELECT COUNT(*) AS Total_Failure
FROM SPACEXTBL
WHERE Mission_Outcome LIKE "%Failure%";

### Analysis 8: Maximum Payload Capacity

I'm curious which booster versions have carried the maximum payload mass.


In [ ]:
%%sql
-- Find boosters that carried the maximum payload mass using a subquery
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);

### Analysis 9: Failed Drone Ship Landings in 2015

Now I'll analyze the failed drone ship landings in 2015, categorized by month.


In [ ]:
%%sql
-- Analyze failed drone ship landings in 2015 by month
SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE substr(Date, 0, 5) = "2015" AND Landing_Outcome = "Failure (drone ship)";

### Analysis 10: Landing Outcome Rankings

Finally, I'll rank the frequency of different landing outcomes between 2010-06-04 and 2017-03-20.


In [ ]:
%%sql
-- Rank landing outcomes by count within a specific date range
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN "2010-06-04" AND "2017-03-20"
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;

## Summary of Findings

Through this SQL analysis, I've gained several insights about SpaceX launches:

1. SpaceX utilizes multiple launch sites, with Cape Canaveral being particularly important
2. The company has conducted numerous missions for NASA's Commercial Resupply Services
3. Different booster versions have varying payload capacities
4. I've identified key milestones like the first successful ground pad landing
5. There's a clear progression in landing success rates over time
6. Certain boosters are capable of handling medium to heavy payloads while still landing successfully

These insights help build a foundation for predicting first stage landing success, which has significant economic implications given the cost savings associated with reusable rockets.